# Data update + hazard screen + hold-out validation (Colab, ready to run)
Run the cells **in order**. Colab has internet, so the hazard screen can query PubChem.
1. **Cell 1** loads the current dataset (102 solvents, built in).
2. **Cell 2 (optional)** automatic hazard screen via PubChem GHS data: lists H-codes and flags CMR codes (H340/H341/H350/H351/H360/H361). It is a *screening aid*: PubChem aggregates supplier/ECHA notifications; confirm important ones in the ECHA C&L Inventory before writing the cmr value.
3. **Cell 3** merges your filled `TODO_primary_values_and_ECHA_check.csv` (upload it): your literature values override the web-table values and the citation goes into `source`.
4. **Cell 4 (hold-out, optional)**: upload `holdout_values.csv` (columns `name,property,literature,source`; properties: alpha,beta,pi_star,dD,dP,dH,eps,bp_C). Those values are removed from training and saved for testing. 
5. Download `curated_dataset_v5.csv`, open the main notebook, set `USE_UPLOADED_CSV=True`, run it, then come back to **Cell 5** with the new `predictions_with_intervals_and_AD.csv` to score the hold-out.

In [ ]:
import pandas as pd, numpy as np, io, re, time, json
from google.colab import files
BASE_CSV = r'''
name,smiles,role,bio_origin,bp_C,flash_C,eps,dD,dP,dH,alpha,beta,pi_star,cmr,source
DMF,CN(C)C=O,reference,fossil,153.0,,36.71,17.4,13.7,11.3,0.0,0.69,0.88,1.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
NMP,CN1CCCC1=O,reference,fossil,,,32.2,18.0,12.3,7.2,0.0,0.77,0.92,1.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr VERIFIED 2026-11: ECHA C&L notified: Repr. 1B H360D; bio_origin: not verified
DMAc,CC(=O)N(C)C,reference,fossil,166.0,,37.78,16.8,11.5,10.2,0.0,0.76,0.88,1.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
DMSO,CS(C)=O,conventional,fossil,189.0,,46.68,18.4,16.4,10.2,0.0,0.76,1.0,0.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Acetonitrile,CC#N,conventional,fossil,,,38.8,15.3,18.0,6.1,0.19,0.4,0.75,0.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Acetone,CC(C)=O,conventional,potential,57.0,,21.4,15.5,10.4,7.0,0.08,0.43,0.71,0.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Sulfolane,O=S1(=O)CCCC1,conventional,fossil,285.0,,44.0,18.0,18.0,9.8,0.0,0.39,0.98,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
THF,C1CCOC1,conventional,potential,,,7.58,16.8,5.7,8.0,0.0,0.55,0.58,1.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Ethyl acetate,CCOC(C)=O,conventional,potential,77.0,,6.4,15.8,5.3,7.2,0.0,0.45,0.55,0.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Dimethyl carbonate,COC(=O)OC,conventional,potential,90.0,,3.17,17.4,9.6,3.9,0.0,0.43,,0.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Toluene,Cc1ccccc1,conventional,fossil,,,2.38,18.0,1.4,2.0,0.0,0.11,0.54,1.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Ethanol,CCO,conventional,full,,,24.5,15.8,8.8,19.4,0.86,0.75,0.54,0.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Methanol,CO,conventional,potential,,,32.6,15.1,12.3,22.3,0.98,0.66,0.6,0.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Dichloromethane,ClCCl,conventional,fossil,,,8.93,18.2,6.3,6.1,0.13,0.1,0.82,1.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Dioxane,C1COCCO1,conventional,fossil,,,2.25,19.0,1.8,7.4,0.0,0.37,0.55,1.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Anisole,COc1ccccc1,conventional,potential,,,4.45,,,,0.0,0.32,0.73,0.0,KT: Stenutz kamlettaft.php; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Cyclohexanone,O=C1CCCCC1,conventional,fossil,,,18.2,17.8,6.3,5.1,0.0,0.53,0.76,0.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
2-Butanone,CCC(C)=O,conventional,potential,80.0,,18.5,16.0,9.0,5.1,0.06,0.48,0.67,0.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Pyridine,c1ccncc1,conventional,fossil,,,12.5,19.0,8.8,5.9,0.0,0.64,0.87,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
gamma-Butyrolactone,O=C1CCCO1,conventional,potential,,,40.96,19.0,16.6,7.4,0.0,0.49,0.87,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Cyrene,C1CC(=O)C2COC1O2,candidate,full,,,,,,,,,,0.0,no sourced property values (to be predicted by QSPR); cmr VERIFIED 2026-11: Sigma-Aldrich SDS 807796 (H319; no CMR class); bio_origin: not verified
GVL,CC1CCC(=O)O1,candidate,full,207.0,,36.14,,,,,,,0.0,"eps: Stenutz solv23.php; bp: Stenutz; cmr VERIFIED 2026-11: Sigma-Aldrich SDS 918660 (H315,H319; no CMR class); bio_origin: not verified"
Propylene carbonate,CC1COC(=O)O1,candidate,potential,,,64.9,20.0,18.0,4.1,0.0,0.4,0.83,0.0,"KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr VERIFIED 2026-11: NITE GHS 108-32-7 (repro/carc/mutagenicity: not classifiable, no data; H319); bio_origin: not verified"
2-MeTHF,CC1CCCO1,candidate,full,79.0,,6.97,,,,0.0,0.45,,0.0,"KT: Stenutz kamlettaft.php; eps: Stenutz solv23.php; bp: Stenutz; cmr VERIFIED 2026-11: Sigma-Aldrich SDS 85488 (H225,H302,H315,H318; no CMR class); bio_origin: not verified"
Ethyl lactate,CC(O)C(=O)OCC,candidate,full,154.0,,13.1,16.0,7.6,12.5,,,,0.0,"Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; bp: Stenutz; cmr VERIFIED 2026-11: Sigma-Aldrich SDS w244007 (H226,H315,H318,H335; no CMR class); bio_origin: not verified"
Dimethyl isosorbide,COC1COC2C1OCC2OC,candidate,full,,,,,,,,,,,no sourced property values (to be predicted by QSPR); cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Glycerol carbonate,OCC1COC(=O)O1,candidate,full,,,,18.0,25.6,17.4,,,,,Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
TMO,CC1(C)CCC(C)(C)O1,candidate,potential,112.0,,5.03,,,,,,,,eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
CPME,COC1CCCC1,candidate,potential,,,,,,,,,,0.0,"no sourced property values (to be predicted by QSPR); cmr VERIFIED 2026-11: Sigma-Aldrich SDS 675970 (H225,H302,H315,H319,H402; no CMR class); bio_origin: not verified"
NBP,CCCCN1CCCC1=O,candidate,potential,,,,,,,,,,,"no sourced property values (to be predicted by QSPR); cmr/bio_origin: not verified here (check ECHA C&L / producer data); NBP cmr left blank: TCI page lists H315,H319 only but SDS unavailable"
PolarClean,COC(=O)C(C)CCC(=O)N(C)C,candidate,fossil,,,,,,,,,,,no sourced property values (to be predicted by QSPR); cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Methyl levulinate,COC(=O)CCC(C)=O,candidate,full,,,,,,,,,,0.0,no sourced property values (to be predicted by QSPR); cmr VERIFIED 2026-11: TCI product page L0044 (H227 only; SDS not on page); bio_origin: not verified
Ethyl levulinate,CCOC(=O)CCC(C)=O,candidate,full,206.0,,11.9,,,,,,,,eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Furfural,O=Cc1ccco1,candidate,full,161.0,,42.0,,,,,,,1.0,eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Dimethyl succinate,COC(=O)CCC(=O)OC,candidate,potential,195.0,,5.1,,,,,,,0.0,"eps: Stenutz solv23.php; bp: Stenutz; cmr VERIFIED 2026-11: TCI product page S0104 (H319,H373; no CMR H-code); bio_origin: not verified"
Diethyl carbonate,CCOC(=O)OCC,candidate,potential,127.0,,3.1,16.6,3.1,6.1,0.0,0.4,0.45,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Triacetin,CC(=O)OCC(COC(C)=O)OC(C)=O,candidate,full,259.0,,7.01,,,,,,,,eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Butyl lactate,CCCCOC(=O)C(C)O,candidate,full,,,,15.6,6.5,10.2,,,,,Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Tetrahydrofurfuryl alcohol,OCC1CCCO1,candidate,full,,,13.48,17.8,8.2,12.9,,,,,Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Dimethyllactamide,CC(O)C(=O)N(C)C,candidate,full,,,,,,,,,,,no sourced property values (to be predicted by QSPR); cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Limonene,CC1=CCC(CC1)C(C)=C,candidate,full,175.0,,2.36,17.2,1.8,4.3,,,,0.0,Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
p-Cymene,Cc1ccc(C(C)C)cc1,candidate,full,177.0,,2.24,,,,,,,,eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Dioxolane,C1COCO1,candidate,potential,,,,18.0,6.5,9.2,0.0,0.45,0.69,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Formamide,NC=O,conventional,fossil,,,111.0,17.2,26.2,19.0,0.71,0.48,0.97,1.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
N-Methylformamide,CNC=O,conventional,fossil,,,171.0,,,,0.62,0.8,0.9,1.0,KT: Stenutz kamlettaft.php; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
"N,N-Dimethylpropionamide",CCC(=O)N(C)C,conventional,fossil,,,33.08,,,,,,,,eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Tetramethylurea,CN(C)C(=O)N(C)C,conventional,fossil,177.0,,24.46,,,,0.0,0.8,0.83,1.0,KT: Stenutz kamlettaft.php; eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
DMPU,CN1CCCN(C)C1=O,conventional,fossil,246.0,,,,,,,,,,bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
"DMI (1,3-dimethylimidazolidinone)",CN1CCN(C)C1=O,conventional,fossil,,,,,,,,,,,no sourced property values (to be predicted by QSPR); cmr/bio_origin: not verified here (check ECHA C&L / producer data)
HMPA,CN(C)P(=O)(N(C)C)N(C)C,conventional,fossil,,,30.0,,,,0.0,1.05,0.87,1.0,KT: Stenutz kamlettaft.php; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
2-Pyrrolidone,O=C1CCCN1,conventional,fossil,,,,,,,0.36,0.77,0.85,1.0,KT: Stenutz kamlettaft.php; cmr VERIFIED 2026-11: Sigma-Aldrich SDS 83300 (H360 Repr. 1B); bio_origin: not verified
NEP,CCN1CCCC1=O,conventional,fossil,,,,,,,,,,1.0,no sourced property values (to be predicted by QSPR); cmr/bio_origin: not verified here (check ECHA C&L / producer data)
NOP,CCCCCCCCN1CCCC1=O,candidate,potential,,,,,,,,,,,no sourced property values (to be predicted by QSPR); cmr/bio_origin: not verified here (check ECHA C&L / producer data)
N-Formylmorpholine,O=CN1CCOCC1,conventional,fossil,,,,,,,,,,,no sourced property values (to be predicted by QSPR); cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Nitromethane,C[N+](=O)[O-],conventional,fossil,,,39.4,15.8,18.8,5.1,0.22,0.06,0.85,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Nitrobenzene,[O-][N+](=O)c1ccccc1,conventional,fossil,,,34.8,20.0,8.6,4.1,0.0,0.3,1.01,1.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Propionitrile,CCC#N,conventional,fossil,,,27.7,,,,0.0,0.39,0.71,,KT: Stenutz kamlettaft.php; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Benzonitrile,N#Cc1ccccc1,conventional,fossil,190.0,,26.0,,,,0.0,0.37,0.9,,KT: Stenutz kamlettaft.php; eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Ethylene carbonate,O=C1OCCO1,conventional,potential,261.0,,,19.4,21.7,5.1,0.0,0.41,,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Acetophenone,CC(=O)c1ccccc1,conventional,fossil,,,17.4,19.6,8.6,3.7,0.04,0.49,0.9,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Cyclopentanone,O=C1CCCC1,conventional,potential,,,14.45,,,,0.0,0.52,0.76,,KT: Stenutz kamlettaft.php; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
MIBK,CC(C)CC(C)=O,conventional,fossil,,,13.11,15.3,6.1,4.1,0.02,0.48,0.65,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Diacetone alcohol,CC(=O)CC(C)(C)O,conventional,fossil,,,18.3,15.8,8.2,10.8,,,,,Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Diethyl ether,CCOCC,conventional,potential,35.0,,4.33,14.5,2.9,5.1,0.0,0.47,0.27,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
MTBE,COC(C)(C)C,conventional,fossil,,,4.5,,,,,,,,eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
"1,2-Dimethoxyethane",COCCOC,conventional,fossil,,,,,,,0.0,0.41,0.53,1.0,KT: Stenutz kamlettaft.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Diglyme,COCCOCCOC,conventional,fossil,,,7.3,,,,0.0,0.4,0.64,1.0,KT: Stenutz kamlettaft.php; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Methyl acetate,COC(C)=O,conventional,potential,57.0,,6.7,15.5,7.2,7.6,0.0,0.42,0.6,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Butyl acetate,CCCCOC(C)=O,conventional,potential,126.0,,5.1,15.8,3.7,6.3,0.0,0.45,0.46,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Isopropyl acetate,CC(C)OC(C)=O,conventional,potential,,,,14.9,4.5,8.2,,,,,Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Hexane,CCCCCC,conventional,fossil,,,1.9,14.9,0.0,0.0,0.0,0.0,-0.04,1.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Heptane,CCCCCCC,conventional,fossil,,,1.92,15.3,0.0,0.0,0.0,0.0,-0.08,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Cyclohexane,C1CCCCC1,conventional,fossil,,,1.88,16.8,0.0,0.2,0.0,0.0,0.0,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Benzene,c1ccccc1,conventional,fossil,,,2.28,18.4,0.0,2.0,0.0,0.1,0.59,1.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
p-Xylene,Cc1ccc(C)cc1,conventional,fossil,,,,17.6,1.0,3.1,,,,,Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Chloroform,ClC(Cl)Cl,conventional,fossil,,,4.81,17.8,3.1,5.7,0.2,0.1,0.58,1.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
"1,2-Dichloroethane",ClCCCl,conventional,fossil,,,,19.0,7.4,4.1,0.0,0.1,0.81,1.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Chlorobenzene,Clc1ccccc1,conventional,fossil,,,5.62,19.0,4.3,2.0,0.0,0.07,0.71,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Isopropanol,CC(C)O,conventional,potential,,,19.92,15.8,6.1,16.4,0.76,0.84,0.48,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
1-Propanol,CCCO,conventional,potential,,,21.8,16.0,6.8,17.4,0.84,0.9,0.52,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
1-Butanol,CCCCO,conventional,potential,,,17.8,16.0,5.7,15.8,0.84,0.84,0.47,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
tert-Butanol,CC(C)(C)O,conventional,potential,,,10.9,15.1,5.1,14.7,0.42,0.93,0.41,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Ethylene glycol,OCCO,conventional,potential,,,37.0,17.0,11.0,26.0,0.9,0.52,0.92,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Propylene glycol,CC(O)CO,conventional,full,,,,16.8,9.4,23.3,,,,,Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
"1,3-Propanediol",OCCCO,candidate,full,,,,,,,,,,,no sourced property values (to be predicted by QSPR); cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Glycerol,OCC(O)CO,candidate,full,,,,17.4,12.1,29.3,,,,,Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
2-Methoxyethanol,COCCO,conventional,fossil,,,16.93,16.2,9.2,16.4,,,,1.0,Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Diethylene glycol,OCCOCCO,conventional,fossil,,,31.7,16.2,14.7,20.5,,,,,Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Water,O,conventional,full,,,80.1,15.6,16.0,42.3,1.17,0.47,1.09,0.0,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Acetic acid,CC(=O)O,conventional,potential,,,6.2,14.5,8.0,13.5,1.12,0.45,0.64,,KT: Stenutz kamlettaft.php; Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Triethylamine,CCN(CC)CC,conventional,fossil,,,3.2,,,,0.0,0.71,0.14,,KT: Stenutz kamlettaft.php; eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Methyl lactate,COC(=O)C(C)O,candidate,full,,,,,,,,,,,no sourced property values (to be predicted by QSPR); cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Levulinic acid,CC(=O)CCC(=O)O,candidate,full,,,,,,,,,,,no sourced property values (to be predicted by QSPR); cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Diethyl succinate,CCOC(=O)CCC(=O)OCC,candidate,full,,,6.64,,,,,,,,eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Dimethyl glutarate,COC(=O)CCCC(=O)OC,candidate,potential,,,,,,,,,,,no sourced property values (to be predicted by QSPR); cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Dimethyl adipate,COC(=O)CCCCC(=O)OC,candidate,potential,,,,16.9,4.3,9.2,,,,,Hansen: Stenutz hansen.php [(cal/mL)^0.5 x2.0455]; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Dimethyl 2-methylglutarate,COC(=O)CCC(C)C(=O)OC,candidate,fossil,,,,,,,,,,,no sourced property values (to be predicted by QSPR); cmr/bio_origin: not verified here (check ECHA C&L / producer data)
Furfuryl alcohol,OCc1ccco1,candidate,full,176.0,,,,,,,,,,bp: Stenutz; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
2-Methylfuran,Cc1ccco1,candidate,full,,,,,,,,,,,no sourced property values (to be predicted by QSPR); cmr/bio_origin: not verified here (check ECHA C&L / producer data); NITE GHS: repro/carc 'classification not possible' (no data) -> cmr left blank
"2,5-Dimethylfuran",Cc1ccc(C)o1,candidate,full,,,,,,,,,,,no sourced property values (to be predicted by QSPR); cmr/bio_origin: not verified here (check ECHA C&L / producer data)
"1,8-Cineole",CC1(C2CCC(O1)(CC2)C)C,candidate,full,,,,,,,,,,,no sourced property values (to be predicted by QSPR); cmr/bio_origin: not verified here (check ECHA C&L / producer data)
alpha-Pinene,CC1=CCC2CC1C2(C)C,candidate,full,,,2.64,,,,,,,,eps: Stenutz solv23.php; cmr/bio_origin: not verified here (check ECHA C&L / producer data)
'''
base = pd.read_csv(io.StringIO(BASE_CSV.strip()))
print(base.shape); base.head()

## Cell 2 (optional): PubChem GHS hazard screen

In [ ]:
import requests
CMR = re.compile(r"H3[45][01]|H36[01]")
def find_h(obj, out):
    if isinstance(obj, dict):
        for v in obj.values(): find_h(v, out)
    elif isinstance(obj, list):
        for v in obj: find_h(v, out)
    elif isinstance(obj, str):
        for m in re.findall(r"H\d{3}[A-Za-z]*", obj): out.add(m)
def ghs(query):
    try:
        r = requests.get(f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{requests.utils.quote(query)}/cids/JSON", timeout=30)
        cid = r.json()["IdentifierList"]["CID"][0]
        v = requests.get(f"https://pubchem.ncbi.nlm.nih.gov/rest/pug_view/data/compound/{cid}/JSON?heading=GHS+Classification", timeout=60)
        if v.status_code != 200: return cid, None
        codes = set(); find_h(v.json(), codes); return cid, sorted(codes)
    except Exception as e:
        return None, f"error: {str(e)[:50]}"
NAMES = None   # None = every solvent whose cmr is not verified; or give a list of names
todo = base[~base.source.str.contains("cmr VERIFIED")].name.tolist() if NAMES is None else NAMES
rows = []
for n in todo:
    cid, codes = ghs(n); time.sleep(0.3)
    if codes is None or isinstance(codes, str): flag, txt = "no data/ERR", str(codes)
    else:
        flag = "CMR code present" if any(CMR.fullmatch(c[:4]) or CMR.match(c) for c in codes) else "no CMR code listed"; txt = " ".join(codes)
    rows.append(dict(name=n, pubchem_cid=cid, hcodes=txt, screen=flag))
H = pd.DataFrame(rows); display(H)
H.to_csv("pubchem_ghs_screen.csv", index=False); files.download("pubchem_ghs_screen.csv")
print("REVIEW before use: 'no CMR code listed' can also mean 'no data'. Confirm in ECHA C&L Inventory and put the final value in the TODO file column TODO_CMR_ECHA... as 0 or 1.")

## Cell 3: merge your filled TODO file

In [ ]:
print("Upload your filled TODO_primary_values_and_ECHA_check.csv  (skip with Cancel to keep the base data)")
up = files.upload()
data = base.copy()
if up:
    T = pd.read_csv(next(iter(up)))
    colmap = {"TODO_alpha_lit":"alpha","TODO_beta_lit":"beta","TODO_pi_star_lit":"pi_star","TODO_dD_lit":"dD","TODO_dP_lit":"dP",
              "TODO_dH_lit":"dH","TODO_bp_C_lit":"bp_C","TODO_flash_C_lit":"flash_C","TODO_eps_lit":"eps"}
    changed = []
    for _, r in T.iterrows():
        m = data.name == r["name"]
        if m.sum() != 1: print("name not found:", r["name"]); continue
        i = data.index[m][0]; cite = str(r.get("value_source_citation_DOI", "")).strip()
        for tc, c in colmap.items():
            v = pd.to_numeric(r.get(tc), errors="coerce")
            if pd.notna(v):
                data.loc[i, c] = v; changed.append((r["name"], c, v))
                data.loc[i, "source"] = f"{c}: {cite or 'user literature value'}; " + str(data.loc[i, "source"])
        cm = pd.to_numeric(r.get("TODO_CMR_ECHA_harmonised_or_notified"), errors="coerce")
        if pd.notna(cm):
            data.loc[i, "cmr"] = cm; data.loc[i, "source"] = str(data.loc[i, "source"]) + f"; cmr {int(cm)} VERIFIED by user (ECHA/SDS) {cite}"
    print(len(changed), "values updated"); display(pd.DataFrame(changed, columns=["name","property","value"]).head(40))
data.to_csv("curated_dataset_v5.csv", index=False); print(data.shape)

## Cell 4 (optional): hold-out values (kept out of training)

In [ ]:
print("Upload holdout_values.csv (name,property,literature,source), or Cancel to skip")
up = files.upload(); train = data.copy(); HOLD = None
if up:
    HOLD = pd.read_csv(next(iter(up)))
    assert {"name","property","literature"} <= set(HOLD.columns)
    for _, r in HOLD.iterrows():
        m = train.name == r["name"]; assert m.sum() == 1, r["name"]
        train.loc[m, r["property"]] = np.nan          # removed from training
    HOLD.to_csv("holdout_values_used.csv", index=False)
    print(len(HOLD), "hold-out values removed from training")
train.to_csv("curated_dataset_v5.csv", index=False); files.download("curated_dataset_v5.csv")
print("Now: main notebook -> set USE_UPLOADED_CSV=True -> run all -> upload its predictions_with_intervals_and_AD.csv in Cell 5")

## Cell 5: score the hold-out (after the main notebook run)

In [ ]:
print("Upload predictions_with_intervals_and_AD.csv from the main run, then holdout_values_used.csv if asked")
up = files.upload()
P = pd.read_csv([k for k in up if "predictions" in k][0])
H2 = pd.read_csv([k for k in up if "holdout" in k][0]) if any("holdout" in k for k in up) else HOLD
P["property"] = P["property"].replace({"eps": "log_eps"})
H2 = H2.copy(); H2["lit_model_scale"] = np.where(H2.property == "eps", np.log10(H2.literature), H2.literature)
H2["property"] = H2.property.replace({"eps": "log_eps"})
R = H2.merge(P, on=["name", "property"], how="left")
R["abs_error"] = (R.pred - R.lit_model_scale).abs()
R["inside_90"] = (R.lit_model_scale >= R.lo) & (R.lit_model_scale <= R.hi)
display(R[["name","property","pred","lo","hi","lit_model_scale","abs_error","inside_90","in_AD"]].round(3))
print("Coverage of 90% intervals:", round(R.inside_90.mean(), 2), "| n =", len(R))
for p, g in R.groupby("property"):
    print(p, "n =", len(g), "RMSE =", round(float(np.sqrt((g.abs_error**2).mean())), 3), "MAE =", round(float(g.abs_error.mean()), 3))
R.to_csv("holdout_results.csv", index=False); files.download("holdout_results.csv")